# Cleaning the Merged Education Dataset

Input: `merged_education_dataset.csv` (raw merge of raw1 + raw2, no cleaning applied yet)
Output: `university_cleaned.csv`

**Workflow:**
1. Load the merged dataset
2. Inspect data quality issues (duplicates, missing values, text inconsistencies)
3. Remove duplicate rows
4. Clean text columns (strip whitespace, normalize casing)
5. Fill missing values (numeric -> median, categorical -> mode / 'Unknown')
6. Final quality check
7. Save `university_cleaned.csv`


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 1. Load the merged dataset

In [2]:
df = pd.read_csv('merged_education_dataset.csv')
print("Shape:", df.shape)
df.head()

Shape: (22145, 38)


,university_id,year,university_name,world_rank,national_rank,overall_score,country,region,city,university_type,academic_reputation_score,employer_reputation_score,citations_score,publications_count,citations_count,citations_per_faculty,h_index,research_output_score,research_productivity_index,subject_field,total_students,international_students_count,international_student_ratio,faculty_count,faculty_to_student_ratio,gender_ratio,female_percentage,male_percentage,degree_level,undergraduate_count,postgraduate_count,country_avg_rank,universities_ranked_count,best_university_rank,country_avg_overall_score,country_avg_academic_reputation,country_avg_citations,country_avg_international_ratio
0,1534,2018,Pontificia Universidad Católica Argentina,364.0,4.0,33.0000,Argentina,Latin America,Buenos Aires,PRIVATE,28.818098,33.421067,33.126557,14526,105384,27.696184,315,31.0,3.82,Social Sciences & Management,15220.0,1063.0,6.984231,3805.0,4.000000,34 : 66,33.991545,66.008455,UNDERGRADUATE & POSTGRADUATE,9560,5660,652.65,17,75.0,33.63,33.37,34.59,83.01
1,1269,2023,National Chi Nan University,1642.0,42.0,15.7275,TAIWAN,Asia,Taiwan,Private,16.417597,17.961854,19.830751,1590,4996,12.776372,72,18.1,4.07,Natural Sciences,5773.0,635.0,11.000000,391.0,14.764706,54:46:00,54.000000,46.000000,Undergraduate & Postgraduate,3452,2321,1041.45,47,77.0,26.27,20.02,19.55,8.87
2,299,2026,Canterbury Christ Church University,1300.5,113.0,29.4325,United Kingdom,Europe,Bexleyheath,Public,6.800000,4.700000,6.800000,3852,9099,6.800609,90,6.8,2.88,Arts & Humanities,23395.0,12867.0,55.000000,1338.0,17.485052,57:43:00,57.000000,43.000000,Undergraduate & Postgraduate,15183,8212,614.48,126,2.0,46.87,35.82,38.92,35.29
3,600,2023,Gazi University,1226.0,28.0,23.7450,TURKEY,Asia,Turkey,Private,22.232035,24.424246,30.888541,6785,110404,36.678974,390,26.6,2.25,Arts & Humanities,41358.0,1241.0,3.000000,3010.0,13.740199,50:50:00,50.000000,50.000000,Undergraduate & Postgraduate,28683,12675,1264.96,73,460.0,24.69,17.31,16.31,8.88
4,2878,2016,University of Oulu,399.0,6.0,35.4300,Finland,Europe,Oulu,Public,43.577886,37.343540,38.804667,2934,15643,22.671226,137,41.2,4.25,Arts & Humanities,14056.0,843.0,6.000000,690.0,20.371014,49:51:00,49.000000,51.000000,Undergraduate & Postgraduate,9480,4576,352.89,9,76.0,38.96,38.59,40.11,9.00


## 2. Inspect data quality

In [3]:
print("Duplicate rows:", df.duplicated().sum())
print()
missing = df.isna().sum()
print("Missing values per column:\n", missing[missing > 0])

Duplicate rows: 20

Missing values per column:
 world_rank                     471
national_rank                  254
employer_reputation_score      222
subject_field                  222
international_student_ratio    221
faculty_count                  221
country_avg_rank                 2
best_university_rank             2
dtype: int64


## 3. Remove duplicate rows

In [4]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"{before} -> {len(df)} rows after dropping {before - len(df)} duplicates")

22145 -> 22125 rows after dropping 20 duplicates


## 4. Clean text columns

Strip stray whitespace and normalize casing on categorical columns so values like 
`" public"`, `"PUBLIC"`, and `"Public"` are treated as the same category.

In [5]:
text_cols = df.select_dtypes(include='object').columns.tolist()
print("Text columns:", text_cols)

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Title-case the categorical (non free-text-name) columns
categorical_cols = ['country', 'region', 'university_type', 'subject_field',
                     'degree_level']
for col in categorical_cols:
    df[col] = df[col].str.title()

Text columns: ['university_name', 'country', 'region', 'city', 'university_type', 'subject_field', 'gender_ratio', 'degree_level']


/tmp/ipykernel_531/3119502531.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include='object').columns.tolist()


## 5. Fill missing values

- Numeric columns: filled with the column **median** (robust to outliers)
- Categorical columns: filled with the column **mode**; if no mode exists, `'Unknown'`


In [6]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in numeric_cols:
    if df[col].isna().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"{col}: filled {df[col].isna().sum()} remaining NaNs with median={median_val:.2f}")

world_rank: filled 0 remaining NaNs with median=725.50
national_rank: filled 0 remaining NaNs with median=20.00
employer_reputation_score: filled 0 remaining NaNs with median=28.01
international_student_ratio: filled 0 remaining NaNs with median=8.00
faculty_count: filled 0 remaining NaNs with median=1162.00
country_avg_rank: filled 0 remaining NaNs with median=709.36
best_university_rank: filled 0 remaining NaNs with median=101.00


In [7]:
for col in text_cols:
    if (df[col] == 'nan').sum() > 0 or df[col].isna().sum() > 0:
        mode_series = df[col].mode()
        fill_val = mode_series.iloc[0] if not mode_series.empty else 'Unknown'
        df[col] = df[col].replace('nan', fill_val)
        df[col] = df[col].fillna(fill_val)
        print(f"{col}: filled missing values with mode='{fill_val}'")

subject_field: filled missing values with mode='Life Sciences & Medicine'


## 6. Fix `gender_ratio`

The raw `gender_ratio` values came in two broken formats: some were mangled into a 
time-like string by spreadsheet auto-formatting (e.g. `"45:55:00"` instead of `"45:55"`), 
and others had inconsistent spacing (e.g. `"36 : 64"`). 

We rebuild `gender_ratio` from the reliable numeric `female_percentage` / `male_percentage` 
columns. **Important:** we use a hyphen (`"45-55"`) instead of a colon (`"45:55"`) as the 
separator. A colon-separated ratio like `45:55` gets auto-reformatted into a time value 
(`45:55:00`) the instant the CSV is opened in Excel/Google Sheets — which is exactly how 
this column got corrupted in the first place. A hyphen avoids that entirely.

In [8]:
print("Before fix, sample gender_ratio values:")
print(df['gender_ratio'].head(10).tolist())

df['gender_ratio'] = (
    df['female_percentage'].round().astype(int).astype(str)
    + '-' +
    df['male_percentage'].round().astype(int).astype(str)
)

print("\nAfter fix, sample gender_ratio values:")
print(df['gender_ratio'].head(10).tolist())

Before fix, sample gender_ratio values:
['34 : 66', '54:46:00', '57:43:00', '50:50:00', '49:51:00', '51:49:00', '48:52:00', '46:54:00', '55:45:00', '67:33:00']



After fix, sample gender_ratio values:
['34-66', '54-46', '57-43', '50-50', '49-51', '51-49', '48-52', '46-54', '55-45', '67-33']


## 7. Final quality check

In [9]:
print("Final shape:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())
print("Remaining missing values:", df.isna().sum().sum())
df.head()

Final shape: (22125, 38)


Remaining duplicates: 0
Remaining missing values: 0


,university_id,year,university_name,world_rank,national_rank,overall_score,country,region,city,university_type,academic_reputation_score,employer_reputation_score,citations_score,publications_count,citations_count,citations_per_faculty,h_index,research_output_score,research_productivity_index,subject_field,total_students,international_students_count,international_student_ratio,faculty_count,faculty_to_student_ratio,gender_ratio,female_percentage,male_percentage,degree_level,undergraduate_count,postgraduate_count,country_avg_rank,universities_ranked_count,best_university_rank,country_avg_overall_score,country_avg_academic_reputation,country_avg_citations,country_avg_international_ratio
0,1534,2018,Pontificia Universidad Católica Argentina,364.0,4.0,33.0000,Argentina,Latin America,Buenos Aires,Private,28.818098,33.421067,33.126557,14526,105384,27.696184,315,31.0,3.82,Social Sciences & Management,15220.0,1063.0,6.984231,3805.0,4.000000,34-66,33.991545,66.008455,Undergraduate & Postgraduate,9560,5660,652.65,17,75.0,33.63,33.37,34.59,83.01
1,1269,2023,National Chi Nan University,1642.0,42.0,15.7275,Taiwan,Asia,Taiwan,Private,16.417597,17.961854,19.830751,1590,4996,12.776372,72,18.1,4.07,Natural Sciences,5773.0,635.0,11.000000,391.0,14.764706,54-46,54.000000,46.000000,Undergraduate & Postgraduate,3452,2321,1041.45,47,77.0,26.27,20.02,19.55,8.87
2,299,2026,Canterbury Christ Church University,1300.5,113.0,29.4325,United Kingdom,Europe,Bexleyheath,Public,6.800000,4.700000,6.800000,3852,9099,6.800609,90,6.8,2.88,Arts & Humanities,23395.0,12867.0,55.000000,1338.0,17.485052,57-43,57.000000,43.000000,Undergraduate & Postgraduate,15183,8212,614.48,126,2.0,46.87,35.82,38.92,35.29
3,600,2023,Gazi University,1226.0,28.0,23.7450,Turkey,Asia,Turkey,Private,22.232035,24.424246,30.888541,6785,110404,36.678974,390,26.6,2.25,Arts & Humanities,41358.0,1241.0,3.000000,3010.0,13.740199,50-50,50.000000,50.000000,Undergraduate & Postgraduate,28683,12675,1264.96,73,460.0,24.69,17.31,16.31,8.88
4,2878,2016,University of Oulu,399.0,6.0,35.4300,Finland,Europe,Oulu,Public,43.577886,37.343540,38.804667,2934,15643,22.671226,137,41.2,4.25,Arts & Humanities,14056.0,843.0,6.000000,690.0,20.371014,49-51,49.000000,51.000000,Undergraduate & Postgraduate,9480,4576,352.89,9,76.0,38.96,38.59,40.11,9.00


## 8. Save the cleaned dataset

In [10]:
df.to_csv('university_cleaned.csv', index=False)
print("Saved university_cleaned.csv with shape:", df.shape)

Saved university_cleaned.csv with shape: (22125, 38)
